### Phase 3: Modellierung (Der Vergleich)

Hier implementieren wir die konkurrierenden Modelle:

#### 1. Hidden-Markov-Models (HMM)
*   **Library:** `hmmlearn.hmm`
*   **Logik:** Unsupervised Learning (Clustering), das Zeitabschnitte mit ähnlichen statistischen Verteilungen gruppiert, um verborgene Marktregimes zu identifizieren.

#### 2. Markov-Switching-Modell (MSM)
*   **Library:** `statsmodels.tsa.regime_switching.markov_regression`
*   **Logik:** Ein statistisches Modell, das Wahrscheinlichkeiten für Regimes berechnet.

#### 3. LSTM-Netzwerk
*   **Library:** `TensorFlow/Keras` oder `PyTorch`.
*   **Architektur:**
    *   Input: Zeitreihen-Fenster (z.B. die letzten 30 Tage der Features).
    *   Layer: LSTM-Layer -> Dropout -> Dense (Softmax).
    
#### 4. Transformer-Netzwerk
*   **Library:** `PyTorch` (`torch.nn.TransformerEncoder`)
*   **Architektur:**
    *   Input: Zeitreihen-Fenster (z.B. die letzten 30 Tage der Features).
    *   Layer: Linear Projection → Positional Encoding → TransformerEncoder (Multi-Head Self-Attention) → Dense (Sigmoid).
*   **Zweck:** Test von Hypothese H2 — Attention-Mechanismus vs. ökonometrische MSM.

Modelle die ein Feedback (gelabelte Daten) benötigen, um Regime zu erkennen, erhalten diese durch das genauste Modell (im Projektverlauf ermittelt) -> Aktuell: Markov-Switching (Univariat)

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
# --- Regime Signal Validation (Helper) ---

BEAR_REGIME = 1
BULL_REGIME = 0

# --- Sanity Check ---
# Wird zu Ende jeder nachfolgenden Model-Cell aufgerufen
def validate_regime_signal(data, model_name: str, auto_invert: bool = True) -> None:
    """
    Standardized sanity check for regime signals.
    Expects {model_name}_Prob and {model_name}_Signal in data.
    """
    prob_col = f'{model_name}_Prob'
    signal_col = f'{model_name}_Signal'
    stats_cols = ['Returns', 'VIX', 'Yield_Spread', prob_col]

    # Regime Statistics
    print(f"\n{'='*60}")
    print(f"   {model_name} — Regime-Statistik")
    print(f"{'='*60}")
    available = [c for c in stats_cols if c in data.columns]
    print(data.groupby(signal_col)[available].mean())
    print(f"\nSignal-Verteilung:\n{data[signal_col].value_counts()}")

    # Plausibility Check
    mean_returns = data.groupby(signal_col)['Returns'].mean()
    if mean_returns.get(BEAR_REGIME, 0) > mean_returns.get(BULL_REGIME, 0):
        print(f"\n WARNUNG: {model_name} Bear-Regime ({BEAR_REGIME}) "
              f"hat höhere Returns als Bull ({BULL_REGIME})!")
        if auto_invert:
            print("    → Labels könnten vertauscht sein. Invertiere:")
            data[signal_col] = 1 - data[signal_col]
            data[prob_col] = 1 - data[prob_col]
            print("    → Labels automatisch invertiert.")
    else:
        print(f"\n{model_name} Plausibilitäts-Check bestanden.")

    # Validierung
    assert prob_col in data.columns, f"{prob_col} fehlt!"
    assert signal_col in data.columns, f"{signal_col} fehlt!"
    assert data[signal_col].isin([BULL_REGIME, BEAR_REGIME]).all(), "Signal enthält ungültige Werte!"
    assert data[signal_col].isna().sum() == 0, "NaN im Signal!"
    assert data[prob_col].between(0, 1).all(), "Prob außerhalb [0,1]!"
    print("Alle formalen Prüfungen bestanden.")

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
df = pd.read_parquet(cfg.data_path("feature_engineered"))

In [ ]:
# =============================================================================
# 1. Hidden Markov Model (HMM)
# Typ: Unsupervised (Clustering)
# Beschreibung: Identifiziert Regime-Cluster über Gaussian-Emissions in
#               Returns, VIX und Yield_Spread ohne gelabelte Daten.
# Config-Key: models.hmm
# =============================================================================

import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

MODEL_NAME = "HMM"

# --- 0. Hyperparameter aus zentraler Config laden ---
hmm_cfg = cfg.models.hmm

# --- 1. Features vorbereiten ---
# Returns (Performance), VIX (Angst) und Yield_Spread (Makro)
hmm_features = hmm_cfg.features
X_hmm = df[hmm_features].values

# --- 2. Skalierung (Standardisierung auf Mittelwert 0 und Varianz 1) ---
scaler_hmm = StandardScaler()
X_hmm_scaled = scaler_hmm.fit_transform(X_hmm)

# --- 3. HMM Modellierung ---
model_hmm = GaussianHMM(
    n_components=hmm_cfg.n_components,
    covariance_type=hmm_cfg.covariance_type,
    n_iter=hmm_cfg.n_iter,
    random_state=hmm_cfg.random_state
)
model_hmm.fit(X_hmm_scaled)

# --- 4. Regimes und Wahrscheinlichkeiten vorhersagen ---
# predict() liefert 0 oder 1
# predict_proba() liefert die Wahrscheinlichkeit für beide Zustände [Prob_0, Prob_1]
hmm_regimes_raw = model_hmm.predict(X_hmm_scaled)
hmm_probs_raw = model_hmm.predict_proba(X_hmm_scaled)

# --- 5. Label-Alignment: Bear (1) = Regime mit höherer Volatilität ---
# Wir definieren Bear (1) als das Regime mit der höheren Volatilität der Renditen.
state_0_vol = df['Returns'][hmm_regimes_raw == 0].std()
state_1_vol = df['Returns'][hmm_regimes_raw == 1].std()

# Wir wollen, dass Regime 1 immer "Bear" ist (höhere Vola)
if state_1_vol > state_0_vol:
    # Fall: Modell-Zustand 1 ist bereits der Bear-Markt
    df[f'{MODEL_NAME}_Prob'] = hmm_probs_raw[:, 1]
else:
    # Fall: Modell-Zustand 0 war eigentlich der Bear-Markt -> wir flippen alles
    df[f'{MODEL_NAME}_Prob'] = hmm_probs_raw[:, 0]

# Signal aus Prob via Threshold ableiten
df[f'{MODEL_NAME}_Signal'] = (df[f'{MODEL_NAME}_Prob'] >= hmm_cfg.threshold).astype(int)

# --- 6. Visualisierung ---
plt.figure(figsize=(15, 3))
plt.fill_between(df.index, 0, 1, where=(df[f'{MODEL_NAME}_Signal'] == 1),
                 color='red', alpha=0.3, label='Bear Market (HMM)')
plt.plot(df.index, df[f'{MODEL_NAME}_Prob'], color='black', alpha=0.2, label='Bear Prob') # Optional: Wahrscheinlichkeitslinie
plt.title(f"{MODEL_NAME} Regimes")
plt.legend()
# HMM Regimes persistieren
plt.savefig(cfg.asset_path("hmm_regimes"), dpi=300, bbox_inches='tight')
plt.show()

# --- Sanity Check ---
validate_regime_signal(df, MODEL_NAME)

In [ ]:
# =============================================================================
# 2. Markov-Switching-Modelle (MS_Univariate & MS_Exo)
# Typ: Ökonometrie (Regression)
# Beschreibung: Zustandsabhängige Regressionsmodelle mit switching variance.
#               MS_Univariate: Nur Returns. MS_Exo: + VIX & Yield_Spread als exogene Variablen.
# Config-Key: models.markov_switching
# =============================================================================

import statsmodels.api as sm
import warnings

# Warnung ignorieren
warnings.filterwarnings("ignore")

# --- 0. Hyperparameter aus zentraler Config laden ---
ms_cfg = cfg.models.markov_switching

# --- 1. Vorbereitung: Index auf Business Days setzen ---
df.index = pd.DatetimeIndex(df.index).to_period('B')

# --- TEIL A: UNIVARIATES MODELL (Baseline) ---
MODEL_NAME_UNI = "MS_Univariate"

# Nur Returns zur Bestimmung von Mittelwert und Varianz
ms_uni_model = sm.tsa.MarkovRegression(
    df['Returns'],
    k_regimes=ms_cfg.k_regimes,
    switching_variance=ms_cfg.switching_variance
)
ms_uni_results = ms_uni_model.fit()

# Label-Alignment: Bear (1) = Regime mit höherer Varianz
# Identifikation des Bärenmarktes (Regime mit der höheren Varianz)
prob_uni_regime_1 = ms_uni_results.smoothed_marginal_probabilities[1]
if ms_uni_results.params['sigma2[1]'] > ms_uni_results.params['sigma2[0]']:
    df[f'{MODEL_NAME_UNI}_Prob'] = prob_uni_regime_1.clip(0, 1)
else:
    df[f'{MODEL_NAME_UNI}_Prob'] = 1 - prob_uni_regime_1.clip(0, 1)

# Signal aus Prob via Threshold ableiten
df[f'{MODEL_NAME_UNI}_Signal'] = (df[f'{MODEL_NAME_UNI}_Prob'] >= ms_cfg.threshold).astype(int)


# --- TEIL B: EXOGENES MODELL (Erweitert) ---
MODEL_NAME_EXO = "MS_Exo"

# Returns als Ziel, VIX und Yield_Spread als erklärende Variablen (exog)
# Hinweis: Die exogenen Variablen beeinflussen hier die Mittelwert-Gleichung der Regimes
exo_vars = df[ms_cfg.exo_features]
ms_exo_model = sm.tsa.MarkovRegression(
    df['Returns'],
    k_regimes=ms_cfg.k_regimes,
    exog=exo_vars,
    switching_variance=ms_cfg.switching_variance
)
ms_exo_results = ms_exo_model.fit()

# Label-Alignment: Bear (1) = Regime mit höherer Varianz
# Identifikation des Bärenmarktes (Regime mit der höheren Varianz)
prob_exo_regime_1 = ms_exo_results.smoothed_marginal_probabilities[1]
if ms_exo_results.params['sigma2[1]'] > ms_exo_results.params['sigma2[0]']:
    df[f'{MODEL_NAME_EXO}_Prob'] = prob_exo_regime_1.clip(0, 1)
else:
    df[f'{MODEL_NAME_EXO}_Prob'] = 1 - prob_exo_regime_1.clip(0, 1)

# Signal aus Prob via Threshold ableiten
df[f'{MODEL_NAME_EXO}_Signal'] = (df[f'{MODEL_NAME_EXO}_Prob'] >= ms_cfg.threshold).astype(int)


# --- ABSCHLUSS ---
# Index wieder zurück in normales Datetime-Format für Plotting
df.index = df.index.to_timestamp()

print("Beide Markov-Modelle erfolgreich berechnet.")

# --- VISUALISIERUNG IM VERGLEICH ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Plot Wahrscheinlichkeiten
ax1.plot(df.index, df[f'{MODEL_NAME_UNI}_Prob'], label='Univariat (Nur Returns)', alpha=0.7)
ax1.plot(df.index, df[f'{MODEL_NAME_EXO}_Prob'], label='Exogen (VIX + Spread)', alpha=0.7, color='red')
ax1.set_title("Vergleich der Bärenmarkt-Wahrscheinlichkeiten")
ax1.legend()

# Plot Signale
ax2.fill_between(df.index, 0, df[f'{MODEL_NAME_UNI}_Signal'], alpha=0.3, label='Signal Univariat')
ax2.step(df.index, df[f'{MODEL_NAME_EXO}_Signal'], color='red', where='post', label='Signal Exogen', alpha=0.8)
ax2.set_title("Vergleich der Handelssignale (1 = Bear/Cash, 0 = Bull/Investiert)")
ax2.legend()

plt.tight_layout()
# Markov-Modelle persistieren
plt.savefig(cfg.asset_path("markov_models"), dpi=300, bbox_inches='tight')
plt.show()

# Kurzer Blick auf das Ergebnis
print(df)

# --- Sanity Check ---
validate_regime_signal(df, MODEL_NAME_UNI)
validate_regime_signal(df, MODEL_NAME_EXO)

In [ ]:
# =============================================================================
# 3. LSTM-Netzwerk (Supervised Regime Classification)
# Typ: Supervised (Labels von MS_Univariate)
# Beschreibung: LSTM-Netzwerk mit rollendem Fenster für zeitreihenbasierte
#               Regime-Klassifikation. Labels stammen aus dem MS_Univariate-Modell.
# Config-Key: models.lstm
# =============================================================================

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

MODEL_NAME = "LSTM"

# --- 0. Hyperparameter aus zentraler Config laden ---
lstm_cfg = cfg.models.lstm

# --- 1. Features vorbereiten ---
# Wir nehmen alle relevanten Informationen für ein "ganzheitliches" Bild
features = cfg.features.model_features
print(f"{MODEL_NAME} nutzt folgende Features: {features}")

# Skalierung
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features])

def create_sequences(data, target, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(target[i])
    return np.array(X), np.array(y)

window_size = lstm_cfg.window_size

# --- 2. Labels und Sequenzen ---
# Wahl der passenden Labels (in config-file)
X, y = create_sequences(scaled_data, df[lstm_cfg.labels].values, window_size)

# Split (Train/Test) - 80% Training, 20% Test
split = int(len(X) * lstm_cfg.train_test_split)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# --- 3. LSTM Architektur ---
model_lstm = Sequential([
    # input_shape passt sich automatisch an die Anzahl der Features an
    LSTM(lstm_cfg.window_size, return_sequences=lstm_cfg.return_sequences, input_shape=(window_size, len(features))),
    Dropout(lstm_cfg.dropout),
    LSTM(lstm_cfg.units),
    Dropout(lstm_cfg.dropout),
    Dense(lstm_cfg.dense, activation=lstm_cfg.activation) # Binäre Klassifikation
])

model_lstm.compile(optimizer=lstm_cfg.optimizer, loss=lstm_cfg.loss, metrics=[lstm_cfg.metrics])

# --- 4. Training ---
print(f"Starte {MODEL_NAME} Training...")
history = model_lstm.fit(
    X_train, y_train,
    epochs=lstm_cfg.epochs,
    batch_size=lstm_cfg.batch_size,
    validation_split=lstm_cfg.validation_split,
    verbose=lstm_cfg.verbose
)

# --- 5. Vorhersagen generieren ---
lstm_probs_raw = model_lstm.predict(X_test)

# --- 6. Test-DataFrame für Backtesting und Visualisierung vorbereiten ---
# Wir schneiden das df so zu, dass es exakt zu den X_test Daten passt
test_df = df.iloc[split + window_size:].copy()

# Wahrscheinlichkeiten und Signale speichern
test_df[f'{MODEL_NAME}_Prob'] = lstm_probs_raw.flatten()
# Signale generieren
test_df[f'{MODEL_NAME}_Signal'] = (test_df[f'{MODEL_NAME}_Prob'] >= lstm_cfg.threshold).astype(int)

# --- 7. Visualisierung ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# A. Wahrscheinlichkeiten
ax1.plot(test_df.index, test_df[f'{MODEL_NAME}_Prob'], color='green', label=f'{MODEL_NAME} Bear Probability')
ax1.axhline(y=0.5, color='red', linestyle='--', label='Threshold 0.5')
ax1.set_title(f"{MODEL_NAME}: Wahrscheinlichkeit für Bärenmarkt")
ax1.legend()

# B. Signale im Vergleich zum Markov-Label (Grundwahrheit)
ax2.plot(test_df.index, test_df['MS_Univariate_Signal'], label='MS_Univariate Target (Ground Truth)', alpha=0.3, color='gray')
ax2.step(test_df.index, test_df[f'{MODEL_NAME}_Signal'], where='post', label=f'{MODEL_NAME} Signal', color='green')
ax2.set_title(f"{MODEL_NAME} Handelssignale (0=Bull, 1=Bear)")
ax2.legend()

plt.tight_layout()
# LSTM-Modell persistieren
plt.savefig(cfg.asset_path("lstm_model"), dpi=300, bbox_inches='tight')
plt.show()

print(f"Finale Test-Genauigkeit: {history.history['val_accuracy'][-1]:.2%}")

print(test_df)

# --- Wir wechseln in diesem Schritt von df auf test_df da sich der Beobachtungszeitraum eingrenzt ---

# --- Sanity Check ---
validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
# =============================================================================
# 4. Unsupervised LSTM (Deep Clustering via Autoencoder + GMM)
# Typ: Unsupervised (keine Labels)
# Beschreibung: LSTM-Autoencoder komprimiert Markt-Sequenzen in latenten Raum;
#               GMM clustert die Embeddings zur Regime-Erkennung.
# Config-Key: models.lstm_unsupervised
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Input

MODEL_NAME = "LSTM_Unsupervised"

# --- 0. Hyperparameter aus zentraler Config laden ---
lu_cfg = cfg.models.lstm_unsupervised

# --- 1. Datenvorbereitung ---
features = cfg.features.model_features
scaler_unsup = StandardScaler()
scaled_data_unsup = scaler_unsup.fit_transform(df[features])

def create_sequences(data, window):
    X = []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
    return np.array(X)

window_size = lu_cfg.window_size
X_all_unsup = create_sequences(scaled_data_unsup, window_size)

# Zeitreihen-Split
split = int(len(X_all_unsup) * lu_cfg.train_test_split)
X_train_u = X_all_unsup[:split]
X_test_u = X_all_unsup[split:]

# --- 2. LSTM-Autoencoder Architektur ---
n_features = len(features)
inputs = Input(shape=(window_size, n_features))

encoder = LSTM(
    lu_cfg.encoder_units,
    activation=lu_cfg.activation,
    return_sequences=False
)(inputs)

decoder = RepeatVector(window_size)(encoder)
decoder = LSTM(
    lu_cfg.decoder_units,
    activation=lu_cfg.activation,
    return_sequences=True
)(decoder)
output = TimeDistributed(Dense(n_features))(decoder)

autoencoder = Model(inputs, output)
autoencoder.compile(
    optimizer=lu_cfg.optimizer,
    loss=lu_cfg.loss
)

# --- 3. Training ---
print(f"Starte Training des {MODEL_NAME}-Autoencoders...")
autoencoder.fit(
    X_train_u, X_train_u,
    epochs=lu_cfg.epochs,
    batch_size=lu_cfg.batch_size,
    validation_split=lu_cfg.validation_split,
    verbose=0
)

# --- 4. Latente Merkmale extrahieren ---
encoder_model = Model(inputs, encoder)
latent_features_test = encoder_model.predict(X_test_u)

# --- 5. Clustering mit GMM ---
gmm = GaussianMixture(
    n_components=lu_cfg.n_components,
    n_init=lu_cfg.gmm_n_init,
    random_state=lu_cfg.gmm_random_state
)
gmm.fit(latent_features_test)

clusters = gmm.predict(latent_features_test)
probs = gmm.predict_proba(latent_features_test)

# --- 6. Label-Alignment: Bear = Cluster mit höherer Volatilität ---
# Wir verknüpfen die Cluster mit den echten Renditen im Test-Zeitraum
temp_results = pd.DataFrame({
    'Returns': df['Returns'].iloc[split + window_size:],
    'Cluster': clusters
})

# Wir berechnen die Standardabweichung (Vola) der Renditen pro Cluster
# Das Cluster mit der höheren Vola definieren wir als Bear (1)
bear_cluster = temp_results.groupby('Cluster')['Returns'].std().idxmax()
print(f"Bear-Regime identifiziert als Cluster: {bear_cluster}")

# --- 7. Ergebnisse in test_df schreiben ---
# Wahrscheinlichkeit für Bärenmarkt
test_df[f'{MODEL_NAME}_Prob'] = probs[:, bear_cluster]
# Binäres Signal (1 = Bear, 0 = Bull)
test_df[f'{MODEL_NAME}_Signal'] = (test_df[f'{MODEL_NAME}_Prob'] >= lu_cfg.threshold).astype(int)

print(f"{MODEL_NAME} abgeschlossen und Signale gespeichert.")

# --- 8. Visualisierung ---
plt.figure(figsize=(15, 5))

# Plot der Wahrscheinlichkeit
plt.subplot(2, 1, 1)
plt.plot(test_df.index, test_df[f'{MODEL_NAME}_Prob'], color='orange', label=f'{MODEL_NAME} Bear Prob')
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
plt.title(f"{MODEL_NAME}: Wahrscheinlichkeit für Bärenmarkt-Regime")
plt.legend()

# Plot der Signale
plt.subplot(2, 1, 2)
plt.fill_between(test_df.index, 0, test_df[f'{MODEL_NAME}_Signal'], color='orange', alpha=0.3, label='Bear Signal')
plt.title(f"{MODEL_NAME}: Handelssignale (1=Bear/Cash)")
plt.legend()

plt.tight_layout()
# Grafik für den Report speichern
plt.savefig(cfg.asset_path("lstm_unsupervised_model"), dpi=300, bbox_inches='tight')
plt.show()

# --- Sanity Check ---
validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
# =============================================================================
# 5. Transformer-Netzwerk (Regime Detection via Multi-Head Self-Attention)
# Typ: Supervised (Labels von MS_Univariate)
# Beschreibung: Transformer-Encoder mit Positional Encoding für zeitreihenbasierte
#               Regime-Klassifikation. Testet Hypothese H2 (Attention > Econometric MSM).
# Config-Key: models.transformer
# =============================================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import math

MODEL_NAME = "Transformer"

# --- 0. Hyperparameter aus zentraler Config laden ---
t_cfg = cfg.models.transformer

# --- 1. Features & Sequenzen vorbereiten ---
features = cfg.features.model_features
print(f"Transformer nutzt folgende Features: {features}")

scaler_transformer = MinMaxScaler()
scaled_data_transformer = scaler_transformer.fit_transform(df[features])

def create_sequences_supervised(data, target, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(target[i])
    return np.array(X), np.array(y)

window_size_t = t_cfg.window_size
X_t, y_t = create_sequences_supervised(
    scaled_data_transformer,
    df[t_cfg.labels].values,
    window_size_t
)

# Train/Test Split (identisch zum LSTM)
split_t = int(len(X_t) * t_cfg.train_test_split)
X_train_t, X_test_t = X_t[:split_t], X_t[split_t:]
y_train_t, y_test_t = y_t[:split_t], y_t[split_t:]

# --- 2. Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# --- 3. Transformer-Klassifikator ---
class TransformerRegimeClassifier(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_layers, dim_feedforward, dropout):
        super().__init__()
        self.input_projection = nn.Linear(n_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x[:, -1, :]
        x = self.classifier(x)
        return x

# --- 4. Modell instanziieren ---
n_features_t = len(features)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_transformer = TransformerRegimeClassifier(
    n_features=n_features_t,
    d_model=t_cfg.d_model,
    n_heads=t_cfg.n_heads,
    n_layers=t_cfg.n_layers,
    dim_feedforward=t_cfg.dim_feedforward,
    dropout=t_cfg.dropout
).to(device)

optimizer_t = torch.optim.Adam(model_transformer.parameters(), lr=t_cfg.learning_rate)

n_pos = y_train_t.sum()           # Anzahl Bear-Samples im Training
n_neg = len(y_train_t) - n_pos    # Anzahl Bull-Samples im Training
raw_weight = n_neg / n_pos
pos_weight = torch.tensor([math.sqrt(raw_weight)], dtype=torch.float32).to(device)
print(f"Class Balance — Bull: {int(n_neg)}, Bear: {int(n_pos)}, "
      f"raw_weight: {raw_weight:.2f}, pos_weight (sqrt): {pos_weight.item():.2f}")
# Erwartet: raw_weight: 3.31, pos_weight (sqrt): ~1.82

criterion_t = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# CEWithLogitsLoss erwartet RAW Logits (kein Sigmoid im Modell-Output!)
model_transformer.classifier = nn.Linear(t_cfg.d_model, 1).to(device)

# --- 5. DataLoader erstellen ---
X_train_tensor = torch.FloatTensor(X_train_t).to(device)
y_train_tensor = torch.FloatTensor(y_train_t).unsqueeze(1).to(device)
X_test_tensor = torch.FloatTensor(X_test_t).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=t_cfg.batch_size, shuffle=True)

# --- 6. Training ---
print("Starte Transformer Training...")
model_transformer.train()
for epoch in range(t_cfg.epochs):
    epoch_loss = 0.0
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        optimizer_t.zero_grad()
        logits = model_transformer(batch_X)          # Raw Logits
        loss = criterion_t(logits, batch_y)
        loss.backward()
        optimizer_t.step()
        epoch_loss += loss.item()

        # Accuracy tracking
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    if t_cfg.verbose and (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / len(train_loader)
        accuracy = correct / total
        print(f"  Epoch {epoch+1}/{t_cfg.epochs} — Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2%}")

# --- 7. Vorhersagen auf Test-Set ---
model_transformer.eval()
with torch.no_grad():
    logits_test = model_transformer(X_test_tensor)
    # Sigmoid erst bei der Inference anwenden (Logits → Probabilities)
    transformer_probs_raw = torch.sigmoid(logits_test).cpu().numpy().flatten()

# --- 8. Ergebnisse in test_df schreiben ---
test_df[f'{MODEL_NAME}_Prob'] = transformer_probs_raw
test_df[f'{MODEL_NAME}_Signal'] = (test_df[f'{MODEL_NAME}_Prob'] >= t_cfg.threshold).astype(int)

# --- 9. Visualisierung ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax1.plot(test_df.index, test_df[f'{MODEL_NAME}_Prob'], color='darkorange', label='Transformer Bear Probability')
ax1.axhline(y=0.5, color='red', linestyle='--', label='Threshold 0.5')
ax1.set_title("Transformer: Wahrscheinlichkeit für Bärenmarkt")
ax1.legend()

ax2.plot(test_df.index, test_df['MS_Univariate_Signal'], label='MS_Univariate Target (Ground Truth)', alpha=0.3, color='gray')
ax2.step(test_df.index, test_df[f'{MODEL_NAME}_Signal'], where='post', label='Transformer Signal', color='darkorange')
ax2.set_title("Transformer Handelssignale (0=Bull, 1=Bear)")
ax2.legend()

plt.tight_layout()
plt.savefig(cfg.asset_path("transformer_model"), dpi=300, bbox_inches='tight')
plt.show()

# --- Sanity Check ---
validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Dynamische Identifikation der Modelle
# Wir suchen alle Spalten, die auf _Signal enden, um die Modellnamen zu extrahieren
model_names = [col.rsplit('_', 1)[0] for col in test_df.columns if col.endswith('_Signal')]

# 2. Farbschema definieren (optional, um Konsistenz zu wahren)
color_map = {
    'MS_Univariate': 'tab:blue',
    'MS_Exo': 'tab:red',
    'HMM': 'tab:purple',
    'LSTM': 'tab:green',
    'LSTM_Unsupervised': 'tab:cyan',
    'Transformer': 'darkorange'
}
# Fallback für neue Modelle, die noch nicht in der Map sind
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# 3. Plot erstellen
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for i, model in enumerate(model_names):
    # Farbe bestimmen
    color = color_map.get(model, default_colors[i % len(default_colors)])
    
    # Linienstil-Logik (z.B. Univariat gestrichelt, Rest durchgezogen)
    ls = '--' if 'Univariate' in model else '-'
    lw = 2.5 if model == 'LSTM' else 1.5
    alpha = 0.8 if model == 'LSTM' else 0.5
    
    # --- Plot 1: Wahrscheinlichkeiten ---
    prob_col = f"{model}_Prob"
    if prob_col in test_df.columns:
        ax1.plot(test_df.index, test_df[prob_col], 
                 label=f"{model.replace('_', ' ')} Prob", 
                 color=color, linestyle=ls, alpha=alpha, linewidth=lw)

    # --- Plot 2: Signale ---
    sig_col = f"{model}_Signal"
    ax2.step(test_df.index, test_df[sig_col], 
             where='post', label=f"{model.replace('_', ' ')} Signal", 
             color=color, linestyle=ls, alpha=alpha, linewidth=lw)

# --- Ax1 Styling ---
ax1.axhline(y=0.5, color='black', linestyle=':', alpha=0.5, label='Schwelle 0.5')
ax1.set_title("Dynamischer Vergleich der Regime-Wahrscheinlichkeiten (Test-Set)")
ax1.set_ylabel("Wahrscheinlichkeit (Bear)")
ax1.legend(loc='upper left', fontsize='small', ncol=2)
ax1.grid(alpha=0.2)
ax1.set_ylim(-0.05, 1.05)

# --- Ax2 Styling ---
ax2.set_title("Dynamischer Vergleich der Handelssignale (0 = Bull, 1 = Bear)")
ax2.set_ylabel("Signal-Status")
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Bulle (0)', 'Bär (1)'])
ax2.legend(loc='upper left', fontsize='small', ncol=2)
ax2.grid(alpha=0.2)
ax2.set_ylim(-0.1, 1.1)

# Layout optimieren
plt.tight_layout()
# Regime Comparison persistieren
plt.savefig(cfg.asset_path("regime_comparison"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
output_path = cfg.data_path("test_data")

# Speichern als Parquet
test_df.to_parquet(output_path)

print(f"Dataframe erfolgreich unter {output_path} gespeichert.")